# 01 Data Preparation

This notebook performs the initial preparation of the ATTICA dataset.

Pipeline

ATTICA Excel
        ↓
Outcome construction
        ↓
Contrastive cohort selection
        ↓
Balanced datasets
        ↓
Missingness analysis
        ↓
Processed datasets

In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat





In [ ]:
from pathlib import Path
import sys
import random
import numpy as np


# ---------------------------------------------------------
# Folder containing the full-cohort notebooks
# ---------------------------------------------------------
EXPERIMENT_DIR = Path.cwd().resolve()

# Main project directory
PROJECT_ROOT = EXPERIMENT_DIR.parents[1]

# ---------------------------------------------------------
# Add the main project directory to Python's import path.
# This allows imports from shared project modules such as
# utils.preprocessing, utils.aggregation, etc.
# ---------------------------------------------------------
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# Shared processed data remain in the main project
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Full-cohort outputs remain isolated inside the experiment
RESULTS_DIR = EXPERIMENT_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
GRAPHS_DIR = RESULTS_DIR / "graphs"

for folder in [
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    GRAPHS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)


print("Project root:", PROJECT_ROOT)
print("Full-cohort results:", RESULTS_DIR)
print("Utils directory:", PROJECT_ROOT / "utils")
print("Utils exists:", (PROJECT_ROOT / "utils").exists())


# ---------------------------------------------------------
# Import shared project utilities
# ---------------------------------------------------------
from utils.preprocessing import *


# Reproducibility
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

In [ ]:
def transform_categorical(df, to_numeric=False):
   #Convert object columns to category, and optionally encode categories as integers.
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype("category")
    if to_numeric:
        cat_cols = df.select_dtypes(include=["category"]).columns
        df[cat_cols] = df[cat_cols].apply(lambda x: x.cat.codes)
    return df

In [ ]:
def load_XLS(
    directory,
    filename,
    to_numeric=True,
    export_csv=False,
):
    """
    Original ATTICA loader.
    """

    df = pd.read_excel(directory / filename)


    df = df.dropna(subset=["Death20", "CVDDeath20", "FU20"]).copy()
    df["Death20"] = (df["Death20"] == "Απεβίωσαν").astype(int)
    df["CVDDeath20"] = (df["CVDDeath20"] == "Yes").astype(int)
   
    df = transform_categorical(
        df,
        to_numeric=to_numeric
    )

    return df    


In [ ]:
def labels_10YM(df):
    """
    10-year mortality label.
    """
    return (
        (df["Death20"] == 1) &
        (df["FU20"] <= 10)
    ).astype(int)

In [ ]:
def contrastive_cohort_10YM(df):
    """
    Keep:

    death within 10 years

    OR

    survival beyond 10 years
    """
    

    mask = (
        ((df["Death20"] == 1) & (df["FU20"] <= 10))
        |
        ((df["Death20"] == 0) & (df["FU20"] > 10))
    )

    return df[mask].copy() 

In [ ]:
df = load_XLS(
    directory=RAW_DIR,
    filename="ATTICA_Study_20_yr_FU_Data_Analysis_Shared.xls",
    to_numeric=False,
)

print(df.shape)
cols = df.columns.tolist()

COHORT = contrastive_cohort_10YM(df)
LABELS = labels_10YM(COHORT)
LABEL = "CVD_mortality10"
print("Full dataset:", df.shape)
print("Selected cohort:", COHORT.shape)
print("Labels:", LABELS.value_counts().to_dict())



In [ ]:
# ============================================================
# AUDIT THE MEDITERRANEAN-DIET TRAJECTORY VARIABLE
# ============================================================

import numpy as np
import pandas as pd

trajectory_audit = df[
    [
        "MedDietScore",
        "MedDietScore5",
        "MedDietScore10",
        "traject_MedDi",
    ]
].copy()

for column in [
    "MedDietScore",
    "MedDietScore5",
    "MedDietScore10",
]:
    trajectory_audit[column] = pd.to_numeric(
        trajectory_audit[column],
        errors="coerce",
    )

trajectory_audit["traject_MedDi"] = (
    trajectory_audit["traject_MedDi"]
    .astype("string")
    .str.strip()
)

# Only baseline and 10-year scores are needed
complete_trajectory = trajectory_audit.dropna(
    subset=[
        "MedDietScore",
        "MedDietScore10",
        "traject_MedDi",
    ]
).copy()


def combine_levels(baseline_level, followup_level):
    return (
        baseline_level.astype(str)
        + "-"
        + followup_level.astype(str)
    )


# ------------------------------------------------------------
# Definition 1: fixed median threshold of 27
# ------------------------------------------------------------

complete_trajectory["baseline_level_27"] = np.where(
    complete_trajectory["MedDietScore"] >= 27,
    "High",
    "Low",
)

complete_trajectory["followup_level_27"] = np.where(
    complete_trajectory["MedDietScore10"] >= 27,
    "High",
    "Low",
)

complete_trajectory["expected_trajectory_27"] = combine_levels(
    complete_trajectory["baseline_level_27"],
    complete_trajectory["followup_level_27"],
)


# ------------------------------------------------------------
# Definition 2: lowest tertile versus middle/upper tertiles
# ------------------------------------------------------------

baseline_tertile_cutoff = float(
    complete_trajectory["MedDietScore"].quantile(1 / 3)
)

followup_tertile_cutoff = float(
    complete_trajectory["MedDietScore10"].quantile(1 / 3)
)

complete_trajectory["baseline_level_tertile"] = np.where(
    complete_trajectory["MedDietScore"]
    <= baseline_tertile_cutoff,
    "Low",
    "High",
)

complete_trajectory["followup_level_tertile"] = np.where(
    complete_trajectory["MedDietScore10"]
    <= followup_tertile_cutoff,
    "Low",
    "High",
)

complete_trajectory["expected_trajectory_tertile"] = (
    combine_levels(
        complete_trajectory[
            "baseline_level_tertile"
        ],
        complete_trajectory[
            "followup_level_tertile"
        ],
    )
)


# ------------------------------------------------------------
# Compare recorded and reconstructed trajectories
# ------------------------------------------------------------

complete_trajectory["matches_threshold_27"] = (
    complete_trajectory["traject_MedDi"]
    == complete_trajectory["expected_trajectory_27"]
)

complete_trajectory["matches_tertile"] = (
    complete_trajectory["traject_MedDi"]
    == complete_trajectory[
        "expected_trajectory_tertile"
    ]
)

print("=" * 90)
print("MEDITERRANEAN-DIET TRAJECTORY AUDIT")
print("=" * 90)

print(
    "Complete trajectory records:",
    len(complete_trajectory),
)

print(
    "\nBaseline lower-tertile cut-off:",
    round(baseline_tertile_cutoff, 3),
)

print(
    "Ten-year lower-tertile cut-off:",
    round(followup_tertile_cutoff, 3),
)

print(
    "\nAgreement using threshold 27:",
    f"{complete_trajectory['matches_threshold_27'].mean():.1%}",
)

print(
    "Agreement using tertile definition:",
    f"{complete_trajectory['matches_tertile'].mean():.1%}",
)

print("\nRecorded trajectory counts:")
print(
    complete_trajectory[
        "traject_MedDi"
    ].value_counts(dropna=False)
)

print("\nRecorded versus threshold-27 reconstruction:")
display(
    pd.crosstab(
        complete_trajectory["traject_MedDi"],
        complete_trajectory[
            "expected_trajectory_27"
        ],
    )
)

print("\nRecorded versus tertile reconstruction:")
display(
    pd.crosstab(
        complete_trajectory["traject_MedDi"],
        complete_trajectory[
            "expected_trajectory_tertile"
        ],
    )
)

print("\nExample inconsistent records:")

display(
    complete_trajectory.loc[
        ~complete_trajectory["matches_tertile"],
        [
            "MedDietScore",
            "MedDietScore5",
            "MedDietScore10",
            "traject_MedDi",
            "expected_trajectory_27",
            "expected_trajectory_tertile",
        ],
    ].head(30)
)

In [ ]:

df_full = COHORT.copy()
#df_full["CVD_mortality10"] = LABELS


df_full[LABEL] = LABELS.to_numpy()


print("="*80)
print("Analytical cohort")
print("="*80)
print("Full dataset:", df.shape)
print("Selected cohort:", COHORT.shape)
print("Labels:", LABELS.value_counts().to_dict())


print(df_full["CVD_mortality10"].value_counts())

In [ ]:
# ============================================================
# DUPLICATE-PARTICIPANT CHECK
# ============================================================

PARTICIPANT_ID = "code"

if PARTICIPANT_ID not in df.columns:
    raise KeyError(
        f"Participant identifier '{PARTICIPANT_ID}' "
        "was not found."
    )

missing_id_count = df[PARTICIPANT_ID].isna().sum()
duplicate_mask = df[PARTICIPANT_ID].duplicated(keep=False)

print("Missing participant identifiers:", missing_id_count)
print("Duplicated participant rows:", duplicate_mask.sum())

if missing_id_count > 0:
    raise ValueError(
        "Missing participant identifiers were detected."
    )

if duplicate_mask.any():
    display(
        df.loc[duplicate_mask]
        .sort_values(PARTICIPANT_ID)
    )

    raise ValueError(
        "Duplicated participant identifiers were detected."
    )

print("Participant identifiers are complete and unique.")

In [ ]:
print(df["Death20"].value_counts(dropna=False))

In [ ]:
print(df["FU20"].describe())

In [ ]:
# ============================================================
# RAW MEDDIETSCORE SENSITIVITY ANALYSIS — NOTEBOOK 01
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm

# ------------------------------------------------------------
# 1. Select the full eligible cohort dataframe
# ------------------------------------------------------------
# Replace df_full_cohort only if your Notebook 01 uses another name.
#
# Common possibilities:
# df_full_cohort
# full_cohort
# cohort_10YM
# df_eligible
#
# The dataframe must contain:
# MedDietScore, Y10M, Age and preferably sex.

analysis_source = df_full.copy()

# ------------------------------------------------------------
# 2. Find the correct column names
# ------------------------------------------------------------

print("Available columns related to diet, outcome, age and sex:")

keywords = [
    "meddiet",
    "diet",
    "y10",
    "death",
    "age",
    "sex",
    "gender",
]

matching_columns = [
    column
    for column in analysis_source.columns
    if any(
        keyword in column.lower()
        for keyword in keywords
    )
]

print(matching_columns)

# ------------------------------------------------------------
# 3. Set column names
# ------------------------------------------------------------

MEDDIET_COLUMN = "MedDietScore"
OUTCOME_COLUMN = "CVD_mortality10"
AGE_COLUMN = "age"

# Change this only if your sex column uses another name.
SEX_COLUMN = "Sex"

required_columns = [
    MEDDIET_COLUMN,
    OUTCOME_COLUMN,
    AGE_COLUMN,
]

missing_required = [
    column
    for column in required_columns
    if column not in analysis_source.columns
]

if missing_required:
    raise ValueError(
        "These required columns were not found:\n"
        f"{missing_required}\n\n"
        "Use the printed column list above and update "
        "MEDDIET_COLUMN, OUTCOME_COLUMN and AGE_COLUMN."
    )

# Sex is optional.
use_sex = SEX_COLUMN in analysis_source.columns

# ------------------------------------------------------------
# 4. Construct the raw analysis dataframe
# ------------------------------------------------------------

model_columns = {
    "MedDietScore": MEDDIET_COLUMN,
    "Outcome": OUTCOME_COLUMN,
    "Age": AGE_COLUMN,
}

if use_sex:
    model_columns["Sex"] = SEX_COLUMN

raw_model_df = pd.DataFrame({
    new_name: pd.to_numeric(
        analysis_source[old_name],
        errors="coerce",
    )
    for new_name, old_name in model_columns.items()
})

raw_model_df = (
    raw_model_df
    .replace(
        [-1, -9, 9, 99, 999, 9999],
        np.nan,
    )
    .dropna()
    .reset_index(drop=True)
)

raw_model_df["Outcome"] = (
    raw_model_df["Outcome"]
    .astype(int)
)

# ------------------------------------------------------------
# 5. Validate the outcome
# ------------------------------------------------------------

unique_outcomes = sorted(
    raw_model_df["Outcome"].unique()
)

if not set(unique_outcomes).issubset({0, 1}):
    raise ValueError(
        "Outcome must be binary 0/1.\n"
        f"Observed values: {unique_outcomes}"
    )

print("\n" + "=" * 90)
print("RAW MEDDIETSCORE ANALYSIS DATA")
print("=" * 90)

print("Participants:", len(raw_model_df))
print("Mortality events:", int(raw_model_df["Outcome"].sum()))
print("MedDietScore range:", (
    raw_model_df["MedDietScore"].min(),
    raw_model_df["MedDietScore"].max(),
))
print("Adjustment variables:", [
    column
    for column in ["Age", "Sex"]
    if column in raw_model_df.columns
])

# ------------------------------------------------------------
# 6. Crude model
# ------------------------------------------------------------

X_crude = sm.add_constant(
    raw_model_df[["MedDietScore"]],
    has_constant="add",
)

crude_model = sm.GLM(
    raw_model_df["Outcome"],
    X_crude,
    family=sm.families.Binomial(),
).fit(
    cov_type="HC3"
)

# ------------------------------------------------------------
# 7. Adjusted model
# ------------------------------------------------------------

adjustment_variables = ["Age"]

if use_sex:
    adjustment_variables.append("Sex")

adjusted_variables = [
    "MedDietScore",
    *adjustment_variables,
]

X_adjusted = sm.add_constant(
    raw_model_df[adjusted_variables],
    has_constant="add",
)

adjusted_model = sm.GLM(
    raw_model_df["Outcome"],
    X_adjusted,
    family=sm.families.Binomial(),
).fit(
    cov_type="HC3"
)

# ------------------------------------------------------------
# 8. Extract odds ratios
# ------------------------------------------------------------

def extract_or(model, variable):
    beta = float(model.params[variable])
    ci = model.conf_int().loc[variable]

    return {
        "Beta": beta,
        "OR": float(np.exp(beta)),
        "CI_Lower": float(np.exp(ci.iloc[0])),
        "CI_Upper": float(np.exp(ci.iloc[1])),
        "P_Value": float(model.pvalues[variable]),
    }


crude_result = extract_or(
    crude_model,
    "MedDietScore",
)

adjusted_result = extract_or(
    adjusted_model,
    "MedDietScore",
)

results_table = pd.DataFrame([
    {
        "Model": "Crude",
        **crude_result,
    },
    {
        "Model": "Adjusted for " + " + ".join(
            adjustment_variables
        ),
        **adjusted_result,
    },
])

print("\n" + "=" * 90)
print("RAW MEDDIETSCORE LOGISTIC MODELS")
print("=" * 90)

display(results_table)

print("\nAdjusted model summary:\n")
print(adjusted_model.summary())

# ------------------------------------------------------------
# 9. Quartile-standardized comparison
# ------------------------------------------------------------

Q1 = float(
    raw_model_df["MedDietScore"].quantile(0.25)
)

Q3 = float(
    raw_model_df["MedDietScore"].quantile(0.75)
)

data_q1 = raw_model_df.copy()
data_q3 = raw_model_df.copy()

data_q1["MedDietScore"] = Q1
data_q3["MedDietScore"] = Q3

X_q1 = sm.add_constant(
    data_q1[adjusted_variables],
    has_constant="add",
)

X_q3 = sm.add_constant(
    data_q3[adjusted_variables],
    has_constant="add",
)

risk_q1 = float(
    adjusted_model.predict(X_q1).mean()
)

risk_q3 = float(
    adjusted_model.predict(X_q3).mean()
)

risk_difference = risk_q3 - risk_q1

risk_ratio = (
    risk_q3 / risk_q1
    if risk_q1 > 0
    else np.nan
)

beta = float(
    adjusted_model.params["MedDietScore"]
)

or_q3_vs_q1 = float(
    np.exp(
        beta * (Q3 - Q1)
    )
)

print("\n" + "=" * 90)
print("STANDARDIZED RAW MEDDIETSCORE EFFECT")
print("=" * 90)

print(f"Participants: {len(raw_model_df)}")
print(
    f"Mortality events: "
    f"{int(raw_model_df['Outcome'].sum())}"
)

print(f"\nLower adherence, Q1: {Q1:.6f}")
print(f"Higher adherence, Q3: {Q3:.6f}")
print(f"Treatment contrast: {Q3 - Q1:.6f}")

print(
    f"\nStandardized mortality risk at Q1: "
    f"{risk_q1:.4%}"
)

print(
    f"Standardized mortality risk at Q3: "
    f"{risk_q3:.4%}"
)

print(
    f"\nRisk difference, Q3 − Q1: "
    f"{risk_difference:.6f}"
)

print(
    f"Risk ratio, Q3 / Q1: "
    f"{risk_ratio:.4f}"
)

print(
    f"Adjusted odds ratio, Q3 vs Q1: "
    f"{or_q3_vs_q1:.4f}"
)

# ------------------------------------------------------------
# 10. Save results
# ------------------------------------------------------------

raw_meddiet_results = pd.DataFrame([
    {
        "Participants": len(raw_model_df),
        "Mortality_Events": int(
            raw_model_df["Outcome"].sum()
        ),
        "Adjustment_Set": ", ".join(
            adjustment_variables
        ),
        "MedDietScore_Q1": Q1,
        "MedDietScore_Q3": Q3,
        "Risk_Q1": risk_q1,
        "Risk_Q3": risk_q3,
        "Risk_Difference": risk_difference,
        "Risk_Ratio": risk_ratio,
        "Adjusted_OR_Q3_vs_Q1": or_q3_vs_q1,
        "Adjusted_OR_Per_Point": (
            adjusted_result["OR"]
        ),
        "OR_CI_Lower_Per_Point": (
            adjusted_result["CI_Lower"]
        ),
        "OR_CI_Upper_Per_Point": (
            adjusted_result["CI_Upper"]
        ),
        "P_Value": (
            adjusted_result["P_Value"]
        ),
    }
])

RAW_MEDDIET_RESULT_FILE = (
    TABLES_DIR
    / "raw_meddietscore_sensitivity.csv"
)

raw_meddiet_results.to_csv(
    RAW_MEDDIET_RESULT_FILE,
    index=False,
)

print("\nSaved to:")
print(RAW_MEDDIET_RESULT_FILE)

In [ ]:
def calculate_completeness(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate the percentage of non-missing observations
    for every variable.
    """
    if df.empty:
        raise ValueError(
            "Cannot calculate completeness for an empty dataframe."
        )

    completeness = df.notna().mean().mul(100)

    return (
        completeness
        .rename("CompletenessPercent")
        .reset_index()
        .rename(columns={"index": "Variable"})
        .sort_values(
            "CompletenessPercent",
            ascending=False
        )
        .reset_index(drop=True)
    )

In [ ]:
# ============================================================
# REFERENCE-COMPATIBLE COMPLETENESS FILTER
# ============================================================

COMPLETENESS_THRESHOLD = 50.0

predictor_df = df_full.drop(
    columns=[LABEL],
    errors="ignore",
)

completeness_table = calculate_completeness(
    predictor_df
)

completeness_table["Keep"] = (
    completeness_table["CompletenessPercent"]
    >= COMPLETENESS_THRESHOLD
)

kept_variables = completeness_table.loc[
    completeness_table["Keep"],
    "Variable",
].tolist()

removed_variables = completeness_table.loc[
    ~completeness_table["Keep"]
].copy()

print("Threshold:", COMPLETENESS_THRESHOLD)
print("Variables evaluated:", len(completeness_table))
print("Variables retained:", len(kept_variables))
print("Variables removed:", len(removed_variables))

In [ ]:
near_threshold = (
    completeness_table.loc[
        completeness_table["CompletenessPercent"].between(35, 55)
    ]
    .sort_values("CompletenessPercent")
)

display(near_threshold)

In [ ]:
variables_to_check = [
    "Alcohol_daily",
    "NIDDM",
    "sex",          # lowercase in the ATTICA dataset
]

display(
    completeness_table.loc[
        completeness_table["Variable"].isin(variables_to_check),
        ["Variable", "CompletenessPercent", "Keep"],
    ]
)

In [ ]:
important_domain_variables = {
    # Demographics
    "age", "Sex",

    # SES
    "Education", "financia", "SES",

    # Behavioral
    "Alcohol_daily", "PhysAct", "PhysAct_level",
    "Smoking", "smoking_ever", "cig_day",
    "pck_years", "typcigar", "year_smo",

    # Treatment
    "MedDietScore",

    # Obesity
    "BMI", "waist", "Waist_Height", "w_t_hr",

    # Metabolic
    "MetS_IDF", "mets", "glucose", "insulin",
    "homa_r", "TyG_index", "LAP",

    # Biomarkers
    "CRP", "Fibrinogen", "Cholesterol",
    "LDL", "HDL", "TGL", "ApoA1", "ApoB",

    # Conditions
    "DM", "NIDDM", "HTN", "HCHOL",
}

retained_set = set(kept_variables)

missing_important = sorted(
    important_domain_variables - retained_set
)

print("Important variables removed:")
print(missing_important)

In [ ]:
# ============================================================
# RAW RANGE AND CONSISTENCY AUDIT
# ============================================================

RANGE_RULES = {
    "age": (18, 100),
    "height": (100, 250),     # centimetres
    "weight": (30, 300),      # kilograms
    "BMI": (10, 80),
    "waist": (30, 250),       # centimetres
    "hip": (30, 250),         # centimetres
    "sbp": (50, 300),
    "dbp": (30, 200),
    "cig_day": (0, 150),
    "year_smo": (0, 90),
    "pck_years": (0, 5000),
}

range_issue_rows = []

for variable, (minimum, maximum) in RANGE_RULES.items():

    if variable not in df.columns:
        continue

    numeric = pd.to_numeric(
        df[variable],
        errors="coerce",
    )

    # Remove unambiguous special-missing codes
    numeric = numeric.replace(
        [-1, -9, 999, 9999],
        np.nan,
    )

    invalid_mask = (
        numeric.notna()
        & ~numeric.between(minimum, maximum)
    )

    for row_index in df.index[invalid_mask]:
        range_issue_rows.append({
            "CheckType": "range",
            "RowIndex": row_index,
            "Variable": variable,
            "Value": numeric.loc[row_index],
            "ExpectedRule": (
                f"{minimum} <= value <= {maximum}"
            ),
        })

In [ ]:
# Systolic blood pressure should not be below diastolic
if {"sbp", "dbp"}.issubset(df.columns):

    sbp = pd.to_numeric(
        df["sbp"],
        errors="coerce",
    )

    dbp = pd.to_numeric(
        df["dbp"],
        errors="coerce",
    )

    inconsistent_bp = (
        sbp.notna()
        & dbp.notna()
        & (sbp < dbp)
    )

    for row_index in df.index[inconsistent_bp]:
        range_issue_rows.append({
            "CheckType": "consistency",
            "RowIndex": row_index,
            "Variable": "sbp/dbp",
            "Value": (
                f"sbp={sbp.loc[row_index]}, "
                f"dbp={dbp.loc[row_index]}"
            ),
            "ExpectedRule": "sbp >= dbp",
        })


# Compare recorded BMI with BMI calculated from weight and height
if {"BMI", "weight", "height"}.issubset(df.columns):

    bmi = pd.to_numeric(
        df["BMI"],
        errors="coerce",
    )

    weight = pd.to_numeric(
        df["weight"],
        errors="coerce",
    )

    height = pd.to_numeric(
        df["height"],
        errors="coerce",
    )

    valid_measurements = (
        bmi.notna()
        & weight.notna()
        & height.notna()
        & (height > 0)
        & (weight > 0)
    )

    calculated_bmi = (
        weight / ((height / 100) ** 2)
    )

    inconsistent_bmi = (
        valid_measurements
        & ((bmi - calculated_bmi).abs() > 3)
    )

    for row_index in df.index[inconsistent_bmi]:
        range_issue_rows.append({
            "CheckType": "consistency",
            "RowIndex": row_index,
            "Variable": "BMI",
            "Value": (
                f"recorded={bmi.loc[row_index]:.2f}, "
                f"calculated={calculated_bmi.loc[row_index]:.2f}"
            ),
            "ExpectedRule": (
                "Absolute difference between recorded and "
                "calculated BMI <= 3"
            ),
        })


# Never-smokers should not have positive smoking exposure
smoking_exposure_columns = [
    column
    for column in [
        "cig_day",
        "year_smo",
        "pck_years",
    ]
    if column in df.columns
]

if (
    "smoking_ever" in df.columns
    and smoking_exposure_columns
):

    smoking_ever = pd.to_numeric(
        df["smoking_ever"],
        errors="coerce",
    )

    for exposure_variable in smoking_exposure_columns:

        exposure = pd.to_numeric(
            df[exposure_variable],
            errors="coerce",
        )

        inconsistent_smoking = (
            (smoking_ever == 0)
            & exposure.notna()
            & (exposure > 0)
        )

        for row_index in df.index[inconsistent_smoking]:
            range_issue_rows.append({
                "CheckType": "consistency",
                "RowIndex": row_index,
                "Variable": exposure_variable,
                "Value": exposure.loc[row_index],
                "ExpectedRule": (
                    "Never-smokers should have zero or "
                    "missing smoking exposure"
                ),
            })

In [ ]:
range_consistency_audit = pd.DataFrame(
    range_issue_rows,
    columns=[
        "CheckType",
        "RowIndex",
        "Variable",
        "Value",
        "ExpectedRule",
    ],
)

if range_consistency_audit.empty:
    print(
        "No raw range or consistency problems detected."
    )
else:
    print(
        "Range or consistency observations requiring review:",
        len(range_consistency_audit),
    )

    display(range_consistency_audit.head(30))

range_consistency_audit.to_csv(
    TABLES_DIR / "raw_range_consistency_audit.csv",
    index=False,
)

In [ ]:
def build_balanced_dataset(df, ratio):

    died = df[df["CVD_mortality10"] == 1]

    survived = df[df["CVD_mortality10"] == 0]

    survivor_n = min(
        len(survived),
        ratio * len(died)
    )

    survived_sample = survived.sample(
        n=survivor_n,
        random_state=RANDOM_STATE
    )

    balanced = pd.concat(
        [died, survived_sample]
    ).sample(
        frac=1,
        random_state=RANDOM_STATE
    )

    return balanced.copy()

In [ ]:
datasets = {}

for ratio in [1,2,4]:

    datasets[ratio] = build_balanced_dataset(
        df_full,
        ratio
    )

    print()

    print(f"Ratio 1:{ratio}")

    print(datasets[ratio].shape)

    print(
        datasets[ratio]["CVD_mortality10"]
        .value_counts()
    )

In [ ]:
missing = (
    df_full
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_table = pd.DataFrame({

    "Variable":missing.index,

    "MissingPercent":missing.values

})

display(missing_table.head(20))

In [ ]:
# ============================================================
# APPLY REFERENCE COMPLETENESS FILTER
# ============================================================

filtered_datasets = {}

for ratio, balanced_df in datasets.items():

    available_variables = [
        variable
        for variable in kept_variables
        if variable in balanced_df.columns
    ]

    filtered_df = balanced_df[
        available_variables
    ].copy()

    # Append the outcome as the final column
    filtered_df[LABEL] = balanced_df[LABEL].values

    filtered_datasets[ratio] = filtered_df

    print(
        f"Filtered 1:{ratio}:",
        filtered_df.shape,
    )

In [ ]:
for ratio, filtered_df in filtered_datasets.items():

    output_path = (
        PROCESSED_DIR
        / f"attica_balanced_1to{ratio}_filtered.csv"
    )

    filtered_df.to_csv(
        output_path,
        index=False,
    )

    print(
        f"Saved filtered 1:{ratio}:",
        output_path,
    )

completeness_table.to_csv(
    PROCESSED_DIR / "variable_completeness.csv",
    index=False,
)

removed_variables.to_csv(
    PROCESSED_DIR
    / "variables_removed_by_completeness.csv",
    index=False,
)

In [ ]:
# ============================================================
# EXPORT FULL NATURALLY IMBALANCED CONTRASTIVE COHORT
# ============================================================

available_full_variables = [
    variable
    for variable in kept_variables
    if variable in df_full.columns
]

df_full_filtered = df_full[
    available_full_variables
].copy()

# Append the binary outcome once, as the final column
df_full_filtered[LABEL] = (
    df_full[LABEL]
    .astype(int)
    .to_numpy()
)

FULL_COHORT_FILE = (
    PROCESSED_DIR
    / "attica_full_contrastive_filtered.csv"
)

df_full_filtered.to_csv(
    FULL_COHORT_FILE,
    index=False,
)

print("=" * 90)
print("FULL NATURALLY IMBALANCED COHORT")
print("=" * 90)

print("Shape:", df_full_filtered.shape)

print(
    "Outcome distribution:",
    df_full_filtered[LABEL]
    .value_counts()
    .sort_index()
    .to_dict(),
)

print(
    "Mortality percentage:",
    round(
        df_full_filtered[LABEL].mean() * 100,
        3,
    ),
)

print("\nSaved to:")
print(FULL_COHORT_FILE)

assert len(df_full_filtered) == 1833
assert int(df_full_filtered[LABEL].sum()) == 97